# 02 · Explore the MBTA API for one bus route
Schedules, Predictions and Routes for a route that appears in the files (route `01` in the
files is `1` in the API). Small requests, status code first.

In [9]:
import os
from datetime import date

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv("../.env")
BASE = "https://api-v3.mbta.com"
HEADERS = {"x-api-key": os.environ["MBTA_API_KEY"]} if os.environ.get("MBTA_API_KEY") else {}
TODAY = date.today().isoformat()

## Routes: the id, short name and direction names

In [10]:
r = requests.get(f"{BASE}/routes/1", headers=HEADERS)
print(r.status_code)
attrs = r.json()["data"]["attributes"]
attrs["short_name"], attrs["direction_names"]

200


('1', ['Outbound', 'Inbound'])

**Notes:** which API direction_id is Inbound?

- `direction_names` is a list indexed by `direction_id`: position 0 is “Outbound”, position 1 is “Inbound”
- So `direction_id` 1 = Inbound and 0 = Outbound, which matches the files’ words

In [11]:
r = requests.get(f"{BASE}/routes", params={"filter[type]": "3"}, headers=HEADERS)
routes = pd.DataFrame([{"id": d["id"], "short_name": d["attributes"]["short_name"]} for d in r.json()["data"]])
routes[routes["short_name"].str.startswith(("SL", "CT"))]

,id,short_name
0,741,SL1
1,742,SL2
2,743,SL3
3,751,SL4
4,749,SL5
5,746,SLW
6,708,CT3


**Notes:** how do SL / CT routes map? Which file routes have no API route?

- For these routes the API id is a number that is different from the name riders see (`short_name`), e.g. SL1 is 741 and CT3 is 708, so they need a lookup table, not just stripping zeros
- Numbered routes only need the leading zero removed (01 → 1)

## Schedules: one day of planned service

In [12]:
r = requests.get(f"{BASE}/schedules", params={"filter[route]": "1", "filter[date]": TODAY}, headers=HEADERS)
print(r.status_code, "records:", len(r.json()["data"]))
r.json()["data"][0]

200 records: 5160


{'attributes': {'arrival_time': None,
  'departure_time': '2026-10-14T04:37:00-04:00',
  'direction_id': 0,
  'drop_off_type': 1,
  'pickup_type': 0,
  'stop_headsign': None,
  'stop_sequence': 1,
  'timepoint': True},
 'id': 'schedule-78181030-64-1',
 'relationships': {'added_routes': {'data': []},
  'route': {'data': {'id': '1', 'type': 'route'}},
  'stop': {'data': {'id': '64', 'type': 'stop'}},
  'trip': {'data': {'id': '78181030', 'type': 'trip'}}},
 'type': 'schedule'}

In [13]:
data = r.json()["data"]
sched = pd.DataFrame([{
    "direction_id": d["attributes"]["direction_id"],
    "stop_id": d["relationships"]["stop"]["data"]["id"],
    "trip_id": d["relationships"]["trip"]["data"]["id"],
    "arrival_time": d["attributes"]["arrival_time"],
    "departure_time": d["attributes"]["departure_time"],
    "timepoint": d["attributes"].get("timepoint"),
} for d in data])
print("timepoint rows:", sched["timepoint"].sum(), "of", len(sched))
sched.head()

timepoint rows: 215 of 5160


,direction_id,stop_id,trip_id,arrival_time,departure_time,timepoint
0,0,64,78181030,None,2026-10-14T04:37:00-04:00,True
1,0,1,78181030,2026-10-14T04:37:00-04:00,2026-10-14T04:37:00-04:00,False
2,0,2,78181030,2026-10-14T04:38:00-04:00,2026-10-14T04:38:00-04:00,False
3,0,6,78181030,2026-10-14T04:39:00-04:00,2026-10-14T04:39:00-04:00,False
4,0,10003,78181030,2026-10-14T04:40:00-04:00,2026-10-14T04:40:00-04:00,False


**Notes:** which API field gives each file column? Which file column has no API answer?

`service_date` → the `filter[date]` I requested
`route_id` → `relationships.route.data.id`
`direction_id` → `attributes.direction_id` (1 = Inbound, 0 = Outbound)
`stop_id` → `relationships.stop.data.id`
`half_trip_id` → `relationships.trip.data.id`
`scheduled` → `attributes.departure_time` (or `arrival_time` at the last stop)
No API answer: `actual`, because schedules are only the plan

## Predictions: the live estimate (not an actual)

In [14]:
r = requests.get(f"{BASE}/predictions", params={"filter[route]": "1", "page[limit]": "3"}, headers=HEADERS)
print(r.status_code)
[d["attributes"] for d in r.json()["data"]]

200


[{'arrival_time': None,
  'arrival_uncertainty': None,
  'departure_time': '2026-10-15T00:07:00-04:00',
  'departure_uncertainty': None,
  'direction_id': 1,
  'last_trip': False,
  'revenue': 'REVENUE',
  'schedule_relationship': None,
  'status': None,
  'stop_sequence': 1,
  'trip_headsign': None,
  'update_type': None},
 {'arrival_time': None,
  'arrival_uncertainty': None,
  'departure_time': '2026-10-15T00:00:00-04:00',
  'departure_uncertainty': None,
  'direction_id': 0,
  'last_trip': False,
  'revenue': 'REVENUE',
  'schedule_relationship': None,
  'status': None,
  'stop_sequence': 1,
  'trip_headsign': None,
  'update_type': None},
 {'arrival_time': None,
  'arrival_uncertainty': None,
  'departure_time': '2026-10-14T23:46:00-04:00',
  'departure_uncertainty': None,
  'direction_id': 0,
  'last_trip': False,
  'revenue': 'REVENUE',
  'schedule_relationship': None,
  'status': None,
  'stop_sequence': 1,
  'trip_headsign': None,
  'update_type': None}]

**Notes:** why can't a prediction be used as `actual`?

- A prediction is the API’s estimate of when the bus will arrive, made before it happens, and it keeps changing until the bus gets there
- `actual` is the time the bus really arrived, recorded afterwards; the API doesn’t keep that, so actuals only exist in the historical files
- Using predictions as actual would train the model on guesses instead of real outcomes, and many prediction fields are empty anyway